# Online Retail Dataset — Initial Data Inspection

## Project

AI Customer Intelligence and Retention Platform

## Purpose

This notebook performs an initial inspection of the raw UCI Online Retail dataset.

The raw dataset is not modified during this phase. The purpose is to identify its structure, missing values, duplicates, cancellations, returns, invalid prices and other potential data-quality issues before defining cleaning rules.

## Author

Mulka Mounika

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print("Pandas version:", pd.__version__)

Pandas version: 2.2.2


In [8]:
candidate_paths = [
    Path("/content/online_retail.xlsx"),
    Path("/content/Online Retail.xlsx"),
    Path("data/raw/online_retail.xlsx"),
    Path("../data/raw/online_retail.xlsx"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "The dataset was not found. Upload online_retail.xlsx to Colab "
        "or place it inside data/raw/."
    )

print("Dataset found:", data_path.resolve())

Dataset found: /content/Online Retail.xlsx


In [10]:
try:
    raw_df = pd.read_excel(data_path)
except Exception as exc:
    raise RuntimeError(f"Unable to read the Excel dataset: {exc}") from exc

print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]}")

Rows: 541,909
Columns: 8


In [11]:
raw_df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,"17,850.00",United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,"17,850.00",United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,"17,850.00",United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,"17,850.00",United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,"13,047.00",United Kingdom


In [12]:
raw_df.tail(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541899,581587,22726,ALARM CLOCK BAKELIKE GREEN,4,2011-12-09 12:50:00,3.75,"12,680.00",France
541900,581587,22730,ALARM CLOCK BAKELIKE IVORY,4,2011-12-09 12:50:00,3.75,"12,680.00",France
541901,581587,22367,CHILDRENS APRON SPACEBOY DESIGN,8,2011-12-09 12:50:00,1.95,"12,680.00",France
541902,581587,22629,SPACEBOY LUNCH BOX,12,2011-12-09 12:50:00,1.95,"12,680.00",France
541903,581587,23256,CHILDRENS CUTLERY SPACEBOY,4,2011-12-09 12:50:00,4.15,"12,680.00",France
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,"12,680.00",France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,"12,680.00",France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,"12,680.00",France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,"12,680.00",France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,"12,680.00",France


In [13]:
expected_columns = {
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
}

actual_columns = set(raw_df.columns)

missing_columns = expected_columns - actual_columns
unexpected_columns = actual_columns - expected_columns

print("Missing expected columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

if missing_columns:
    raise ValueError(
        f"The dataset is missing required columns: {sorted(missing_columns)}"
    )

Missing expected columns: set()
Unexpected columns: set()


In [14]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [15]:
schema_summary = pd.DataFrame({
    "Column": raw_df.columns,
    "DataType": raw_df.dtypes.astype(str).values,
    "NonNullRows": raw_df.notna().sum().values,
    "NullRows": raw_df.isna().sum().values,
    "UniqueValues": [
        raw_df[column].nunique(dropna=True)
        for column in raw_df.columns
    ],
})

schema_summary

,Column,DataType,NonNullRows,NullRows,UniqueValues
0,InvoiceNo,object,541909,0,25900
1,StockCode,object,541909,0,4070
2,Description,object,540455,1454,4223
3,Quantity,int64,541909,0,722
4,InvoiceDate,datetime64[ns],541909,0,23260
5,UnitPrice,float64,541909,0,1630
6,CustomerID,float64,406829,135080,4372
7,Country,object,541909,0,38


In [16]:
missing_summary = pd.DataFrame({
    "Column": raw_df.columns,
    "MissingRows": raw_df.isna().sum().values,
})

missing_summary["MissingPercentage"] = (
    missing_summary["MissingRows"] / len(raw_df) * 100
).round(2)

missing_summary = missing_summary.sort_values(
    by="MissingRows",
    ascending=False,
).reset_index(drop=True)

missing_summary

,Column,MissingRows,MissingPercentage
0,CustomerID,135080,24.93
1,Description,1454,0.27
2,StockCode,0,0.00
3,InvoiceNo,0,0.00
4,Quantity,0,0.00
5,InvoiceDate,0,0.00
6,UnitPrice,0,0.00
7,Country,0,0.00


In [17]:
duplicate_mask = raw_df.duplicated(keep=False)

duplicate_row_count = int(raw_df.duplicated().sum())
all_duplicate_occurrences = int(duplicate_mask.sum())

print(f"Duplicate rows after the first occurrence: {duplicate_row_count:,}")
print(f"All rows involved in duplicate groups: {all_duplicate_occurrences:,}")

Duplicate rows after the first occurrence: 5,268
All rows involved in duplicate groups: 10,147


In [18]:
raw_df.loc[duplicate_mask].sort_values(
    by=["InvoiceNo", "StockCode", "InvoiceDate"]
).head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,"17,908.00",United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,"17,908.00",United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,"17,908.00",United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,"17,908.00",United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,"17,908.00",United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,"17,908.00",United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,"17,908.00",United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,"17,908.00",United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,"17,920.00",United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,"17,920.00",United Kingdom


In [19]:
invoice_no = raw_df["InvoiceNo"].astype("string").str.strip()

quantity = pd.to_numeric(
    raw_df["Quantity"],
    errors="coerce",
)

unit_price = pd.to_numeric(
    raw_df["UnitPrice"],
    errors="coerce",
)

invoice_date = pd.to_datetime(
    raw_df["InvoiceDate"],
    errors="coerce",
)

customer_id = raw_df["CustomerID"]

In [20]:
cancellation_mask = invoice_no.str.upper().str.startswith(
    "C",
    na=False,
)

negative_quantity_mask = quantity < 0
zero_quantity_mask = quantity == 0

print(f"Cancelled transaction lines: {cancellation_mask.sum():,}")
print(f"Negative-quantity lines: {negative_quantity_mask.sum():,}")
print(f"Zero-quantity lines: {zero_quantity_mask.sum():,}")

Cancelled transaction lines: 9,288
Negative-quantity lines: 10,624
Zero-quantity lines: 0


In [21]:
cancellation_cross_check = pd.crosstab(
    cancellation_mask,
    negative_quantity_mask,
    rownames=["Invoice starts with C"],
    colnames=["Quantity is negative"],
)

cancellation_cross_check

Quantity is negative,False,True
Invoice starts with C,,
False,531285,1336
True,0,9288


In [22]:
raw_df.loc[cancellation_mask].head(15)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,"14,527.00",United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,"15,311.00",United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,"17,548.00",United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,"17,548.00",United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,"17,548.00",United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,"17,548.00",United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,"17,548.00",United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,"17,548.00",United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,"17,548.00",United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,"17,897.00",United Kingdom


In [23]:
negative_price_mask = unit_price < 0
zero_price_mask = unit_price == 0
missing_price_mask = unit_price.isna()

print(f"Negative-price lines: {negative_price_mask.sum():,}")
print(f"Zero-price lines: {zero_price_mask.sum():,}")
print(f"Missing or non-numeric price lines: {missing_price_mask.sum():,}")

Negative-price lines: 2
Zero-price lines: 2,515
Missing or non-numeric price lines: 0


In [24]:
raw_df.loc[
    negative_price_mask | zero_price_mask | missing_price_mask
].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.00,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.00,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.00,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.00,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.00,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.00,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.00,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.00,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.00,NaN,United Kingdom


In [25]:
print("Earliest transaction:", invoice_date.min())
print("Latest transaction:", invoice_date.max())
print("Invalid or missing dates:", invoice_date.isna().sum())

Earliest transaction: 2010-12-01 08:26:00
Latest transaction: 2011-12-09 12:50:00
Invalid or missing dates: 0


In [26]:
monthly_row_counts = (
    invoice_date
    .dt.to_period("M")
    .value_counts()
    .sort_index()
    .rename_axis("Month")
    .reset_index(name="TransactionLines")
)

monthly_row_counts

,Month,TransactionLines
0,2010-12,42481
1,2011-01,35147
2,2011-02,27707
3,2011-03,36748
4,2011-04,29916
5,2011-05,37030
6,2011-06,36874
7,2011-07,39518
8,2011-08,35284
9,2011-09,50226


In [27]:
unique_summary = pd.DataFrame({
    "Measure": [
        "Unique invoice numbers",
        "Unique product codes",
        "Unique customer identifiers",
        "Unique countries",
    ],
    "Value": [
        invoice_no.nunique(dropna=True),
        raw_df["StockCode"].nunique(dropna=True),
        customer_id.nunique(dropna=True),
        raw_df["Country"].nunique(dropna=True),
    ],
})

unique_summary

,Measure,Value
0,Unique invoice numbers,25900
1,Unique product codes,4070
2,Unique customer identifiers,4372
3,Unique countries,38


In [28]:
raw_df[["Quantity", "UnitPrice"]].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

,Quantity,UnitPrice
count,"541,909.00","541,909.00"
mean,9.55,4.61
std,218.08,96.76
min,"-80,995.00","-11,062.06"
1%,-2.00,0.19
5%,1.00,0.42
25%,1.00,1.25
50%,3.00,2.08
75%,10.00,4.13
95%,29.00,9.95


In [29]:
print("Largest quantities:")
display(
    raw_df.nlargest(10, "Quantity")[
        ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]
    ]
)

print("Most negative quantities:")
display(
    raw_df.nsmallest(10, "Quantity")[
        ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]
    ]
)

print("Highest unit prices:")
display(
    raw_df.nlargest(10, "UnitPrice")[
        ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]
    ]
)

Largest quantities:


,InvoiceNo,StockCode,Description,Quantity,UnitPrice
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.00
74614,542504,37413,NaN,5568,0.00
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,0.21
206121,554868,22197,SMALL POPCORN HOLDER,4300,0.72
220843,556231,85123A,?,4000,0.00
97432,544612,22053,EMPIRE DESIGN ROSETTE,3906,0.82
270885,560599,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,3186,0.06
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10


Most negative quantities:


,InvoiceNo,StockCode,Description,Quantity,UnitPrice
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04
225529,556690,23005,printing smudges/thrown away,-9600,0.00
225530,556691,23005,printing smudges/thrown away,-9600,0.00
4287,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.03
225528,556687,23003,Printing smudges/thrown away,-9058,0.00
115818,546152,72140F,throw away,-5368,0.00
431381,573596,79323W,"Unsaleable, destroyed.",-4830,0.00
341601,566768,16045,NaN,-3667,0.00
323458,565304,16259,NaN,-3167,0.00


Highest unit prices:


,InvoiceNo,StockCode,Description,Quantity,UnitPrice
222681,C556445,M,Manual,-1,"38,970.00"
524602,C580605,AMAZONFEE,AMAZON FEE,-1,"17,836.46"
43702,C540117,AMAZONFEE,AMAZON FEE,-1,"16,888.02"
43703,C540118,AMAZONFEE,AMAZON FEE,-1,"16,453.71"
15016,C537630,AMAZONFEE,AMAZON FEE,-1,"13,541.33"
15017,537632,AMAZONFEE,AMAZON FEE,1,"13,541.33"
16356,C537651,AMAZONFEE,AMAZON FEE,-1,"13,541.33"
16232,C537644,AMAZONFEE,AMAZON FEE,-1,"13,474.79"
524601,C580604,AMAZONFEE,AMAZON FEE,-1,"11,586.50"
299982,A563185,B,Adjust bad debt,1,"11,062.06"


In [30]:
quality_metrics = {
    "Total rows": len(raw_df),
    "Total columns": raw_df.shape[1],
    "Complete duplicate rows": raw_df.duplicated().sum(),
    "Rows with missing CustomerID": customer_id.isna().sum(),
    "Rows with missing Description": raw_df["Description"].isna().sum(),
    "Cancelled transaction lines": cancellation_mask.sum(),
    "Negative-quantity lines": negative_quantity_mask.sum(),
    "Zero-quantity lines": zero_quantity_mask.sum(),
    "Negative-price lines": negative_price_mask.sum(),
    "Zero-price lines": zero_price_mask.sum(),
    "Invalid or missing invoice dates": invoice_date.isna().sum(),
    "Unique invoices": invoice_no.nunique(dropna=True),
    "Unique products": raw_df["StockCode"].nunique(dropna=True),
    "Unique identified customers": customer_id.nunique(dropna=True),
    "Unique countries": raw_df["Country"].nunique(dropna=True),
}

quality_summary = pd.DataFrame(
    quality_metrics.items(),
    columns=["Metric", "Value"],
)

quality_summary["Value"] = pd.to_numeric(
    quality_summary["Value"],
    errors="coerce",
).astype("Int64")

quality_summary

,Metric,Value
0,Total rows,541909
1,Total columns,8
2,Complete duplicate rows,5268
3,Rows with missing CustomerID,135080
4,Rows with missing Description,1454
5,Cancelled transaction lines,9288
6,Negative-quantity lines,10624
7,Zero-quantity lines,0
8,Negative-price lines,2
9,Zero-price lines,2515


In [31]:
if (
    data_path.parent.name == "raw"
    and data_path.parent.parent.name == "data"
):
    project_root = data_path.parents[2]
else:
    project_root = Path.cwd()

outputs_dir = project_root / "outputs"
docs_dir = project_root / "docs"

outputs_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)

quality_summary.to_csv(
    outputs_dir / "data_quality_summary.csv",
    index=False,
)

missing_summary.to_csv(
    outputs_dir / "missing_values_summary.csv",
    index=False,
)

schema_summary.to_csv(
    outputs_dir / "schema_summary.csv",
    index=False,
)

monthly_row_counts.to_csv(
    outputs_dir / "monthly_transaction_line_counts.csv",
    index=False,
)

print("Outputs saved to:", outputs_dir.resolve())

Outputs saved to: /content/outputs


In [32]:
def metric_value(metric_name: str) -> int:
    value = quality_metrics[metric_name]
    return int(value)


report = f"""# Initial Data-Quality Findings

## 1. Inspection Scope

This document records the initial inspection of the raw UCI Online Retail dataset.

The raw data was loaded without deleting, replacing or modifying source records.

## 2. Dataset Structure

- Total rows: {metric_value("Total rows"):,}
- Total columns: {metric_value("Total columns"):,}
- Unique invoice numbers: {metric_value("Unique invoices"):,}
- Unique product codes: {metric_value("Unique products"):,}
- Unique identified customers: {metric_value("Unique identified customers"):,}
- Unique countries: {metric_value("Unique countries"):,}
- Earliest transaction date: {invoice_date.min()}
- Latest transaction date: {invoice_date.max()}

## 3. Missing Values

- Rows with missing CustomerID: {metric_value("Rows with missing CustomerID"):,}
- Rows with missing Description: {metric_value("Rows with missing Description"):,}
- Invalid or missing invoice dates: {metric_value("Invalid or missing invoice dates"):,}

Missing customer identifiers prevent reliable customer-level aggregation. However, these records may remain useful for transaction, revenue or product-level analysis.

No records were removed during inspection.

## 4. Duplicate Records

- Complete duplicate rows after the first occurrence: {metric_value("Complete duplicate rows"):,}

Repeated invoice numbers are not automatically duplicates because one invoice may contain several product lines.

Complete duplicate rows require further investigation before a removal rule is applied.

## 5. Cancellations and Returns

- Cancelled transaction lines: {metric_value("Cancelled transaction lines"):,}
- Negative-quantity lines: {metric_value("Negative-quantity lines"):,}
- Zero-quantity lines: {metric_value("Zero-quantity lines"):,}

Cancelled invoices and negative quantities may represent genuine returns or reversals. They will be separated from completed sales rather than automatically treated as invalid data.

## 6. Price Issues

- Negative-price lines: {metric_value("Negative-price lines"):,}
- Zero-price lines: {metric_value("Zero-price lines"):,}

Zero and negative prices require investigation. They will not be replaced or removed until their likely business meaning has been reviewed.

## 7. Preliminary Quality Risks

The initial inspection identified the following issues requiring cleaning rules:

1. Missing customer identifiers
2. Missing product descriptions
3. Complete duplicate rows
4. Cancelled invoices
5. Negative quantities
6. Zero or negative unit prices
7. Extreme quantity and price values
8. Non-product or administrative stock codes
9. Transactions that may not represent completed retail sales

## 8. Decisions Deferred to the Cleaning Phase

The following decisions have not yet been made:

- Whether complete duplicate rows should be removed
- Whether unidentified-customer transactions should be retained for revenue analysis
- How returns and cancellations should be represented
- How zero-price records should be handled
- Whether administrative stock codes should be excluded
- Whether extreme values represent valid wholesale purchases
- Which records qualify as valid completed sales

These rules will be defined and documented before a processed dataset is created.
"""

report_path = docs_dir / "03-data-quality-findings.md"
report_path.write_text(report, encoding="utf-8")

print("Report saved to:", report_path.resolve())

Report saved to: /content/docs/03-data-quality-findings.md
